In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [4]:
# Load Features
# ======================
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/FYP/features.csv")

X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values


In [5]:
# Encode labels
encoder = OneHotEncoder()
Y = encoder.fit_transform(Y.reshape(-1, 1)).toarray()



In [6]:
# Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, shuffle=True
)



In [7]:
# Scale features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)



In [8]:
# Expand dims for CNN
x_train = np.expand_dims(x_train, axis=2)
x_test = np.expand_dims(x_test, axis=2)


In [9]:
# CNN Model
# ======================
model = Sequential([
    Conv1D(256, 5, padding="same", activation="relu", input_shape=(x_train.shape[1], 1)),
    MaxPooling1D(5, strides=2, padding="same"),

    Conv1D(256, 5, padding="same", activation="relu"),
    MaxPooling1D(5, strides=2, padding="same"),

    Conv1D(128, 5, padding="same", activation="relu"),
    MaxPooling1D(5, strides=2, padding="same"),
    Dropout(0.2),

    Conv1D(64, 5, padding="same", activation="relu"),
    MaxPooling1D(5, strides=2, padding="same"),

    Flatten(),
    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(y_train.shape[1], activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 162, 256)       │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 81, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 81, 256)        │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 41, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 41, 128)        │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 21, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 21, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 21, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 11, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 704)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        22,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           264 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 557,288 (2.13 MB)

 Trainable params: 557,288 (2.13 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Training
# ======================
lr_reduce = ReduceLROnPlateau(monitor="loss", factor=0.4, patience=2)

model.fit(
    x_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[lr_reduce]
)


Epoch 1/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.2558 - loss: 1.8530 - val_accuracy: 0.4135 - val_loss: 1.4636 - learning_rate: 0.0010
Epoch 2/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.3828 - loss: 1.5196 - val_accuracy: 0.4620 - val_loss: 1.3419 - learning_rate: 0.0010
Epoch 3/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.4343 - loss: 1.4065 - val_accuracy: 0.4837 - val_loss: 1.2814 - learning_rate: 0.0010
Epoch 4/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.4653 - loss: 1.3420 - val_accuracy: 0.5055 - val_loss: 1.2252 - learning_rate: 0.0010
Epoch 5/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.4878 - loss: 1.2708 - val_accuracy: 0.5155 - val_loss: 1.1982 - learning_rate: 0.0010
Epoch 6/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.5061 - loss: 1.2448 - val_accuracy: 0.5158 - val_loss: 1.1992 - learning_rate: 0.0010
Epoch 7/50
457/457 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5113 - loss: 1.217

In [11]:
# Save Everything
# ======================
model.save("/content/drive/MyDrive/Colab Notebooks/FYP/audio_emotion_cnn.h5")
joblib.dump(scaler, "/content/drive/MyDrive/Colab Notebooks/FYP/scaler.pkl")
joblib.dump(encoder, "/content/drive/MyDrive/Colab Notebooks/FYP/encoder.pkl")

print("Model and preprocessing objects saved successfully!")

Model and preprocessing objects saved successfully!
